In [ ]:
# ============================================================
# 1.1 数据加载与合并
# CAAI-BDSC2023 社交图谱动态链接预测
# ============================================================
import pandas as pd
import numpy as np
import json
import os
import gc
from pathlib import Path

# 路径配置
BASE_DIR = Path(r"D:\GNN")
INFO_DIR = BASE_DIR / "info"

# 文件路径
# 6个项目数据文件
USER_PATH       = INFO_DIR / "user_info.json"
ITEM_PATH       = INFO_DIR / "item_info.json"
SHARE_TRAIN     = INFO_DIR / "item_share_train_info.json"
SHARE_PRELIM    = INFO_DIR / "item_share_preliminary_test_info.json"
SHARE_FINAL_TR  = INFO_DIR / "item_share_final_train_info.json"
SHARE_FINAL_TE  = INFO_DIR / "item_share_final_test_info.json"

print("✅ 库导入完成，路径配置完毕")

In [3]:
# ============================================================
# 加载用户表 & 商品表 (相对较小，直接读取)
# ============================================================
print("正在加载用户表...")
df_user = pd.read_json(USER_PATH, orient="records")
print(f"  用户表: {df_user.shape[0]:,} 行, {df_user.shape[1]} 列")
print(f"  列名: {list(df_user.columns)}")
display(df_user.head(3))

print("\n正在加载商品表...")
df_item = pd.read_json(ITEM_PATH, orient="records")
print(f"  商品表: {df_item.shape[0]:,} 行, {df_item.shape[1]} 列")
print(f"  列名: {list(df_item.columns)}")
display(df_item.head(3))

正在加载用户表...
  用户表: 115,849 行, 4 列
  列名: ['user_id', 'user_gender', 'user_age', 'user_level']


,user_id,user_gender,user_age,user_level
0,0001b424e69a120a20d2bd52eed56b29,0,3,6
1,00037492f9a71a84356386ded85e895f,0,3,7
2,0004a529dae0579a3bce336496ad11a6,0,4,4



正在加载商品表...
  商品表: 438,164 行, 5 列
  列名: ['item_id', 'cate_id', 'cate_level1_id', 'brand_id', 'shop_id']


,item_id,cate_id,cate_level1_id,brand_id,shop_id
0,00001275731a15de9d534ffc763337ba,50008898,16,29534,155156474
1,000026c7927131157389db76f18c988c,1629,16,0,215977934
2,00005ca1a004d909f61157ab67563c7a,121406022,50022517,38774304,100866814


In [4]:
# ============================================================
# 加载分享行为表 (文件较大，逐文件读取并释放内存)
# 这些都是针对数据集进行行数的先行检查
# ============================================================
def load_share_table(filepath, name):
    """加载单个分享行为 JSON，返回 DataFrame"""
    print(f"加载 {name}... ", end="", flush=True)
    df = pd.read_json(filepath, orient="records")
    print(f"→ {df.shape[0]:,} 条记录, {df.shape[1]} 列")
    print(f"  时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
    return df

df_train       = load_share_table(SHARE_TRAIN,    "训练集")
df_prelim_test = load_share_table(SHARE_PRELIM,   "初赛测试集")
df_final_train = load_share_table(SHARE_FINAL_TR, "决赛训练集")
df_final_test  = load_share_table(SHARE_FINAL_TE, "决赛测试集")

print(f"\n📊 总计: {sum([len(d) for d in [df_train, df_prelim_test, df_final_train, df_final_test]]):,} 条分享记录")

加载 训练集... → 602,679 条记录, 4 列
  时间范围: 2021-12-27 00:00:15 ~ 2022-10-29 02:01:12
加载 初赛测试集... → 118,424 条记录, 4 列
  时间范围: 2022-10-29 02:48:53 ~ 2022-12-06 10:42:08
加载 决赛训练集... → 118,424 条记录, 4 列
  时间范围: 2022-10-29 02:48:53 ~ 2022-12-06 10:42:08
加载 决赛测试集... → 115,414 条记录, 4 列
  时间范围: 2022-12-06 10:42:13 ~ 2023-02-27 23:58:57

📊 总计: 954,941 条分享记录


In [5]:
# ============================================================
# 合并 — 分享表 JOIN 用户表(两侧) JOIN 商品表
# 每条记录展开为：
#   inviter特征 | voter特征(训练集才有) | item特征 | timestamp
# ============================================================

def merge_share_with_features(df_share, df_user, df_item):
    """
    将分享表与用户表(两侧)、商品表做 LEFT JOIN
    - 训练集: 有 voter_id，完整合并两侧用户特征
    - 测试集: 无 voter_id，仅合并 inviter 侧 + 商品特征

    返回合并后的 DataFrame
    """
    # ---- 准备 inviter 侧用户表（重命名列） ----
    df_inviter = df_user.rename(columns={
        "user_id":     "inviter_id",
        "user_gender": "inviter_gender",
        "user_age":    "inviter_age",
        "user_level":  "inviter_level",
    })

    # ---- JOIN inviter 侧 ----
    # '''
    # # merge 写法（当前）
    # df = df_share.merge(df_inviter, on="inviter_id", how="left")
    #
    # # join 等价写法 — 必须先把右表的 join key 设成 index
    # df = df_share.join(df_inviter.set_index("inviter_id"), on="inviter_id", how="left")
    # '''
    df = df_share.merge(df_inviter, on="inviter_id", how="left")

    # ---- JOIN voter 侧 (仅当分享表中有 voter_id 列时) ----
    if "voter_id" in df.columns:
        df_voter = df_user.rename(columns={
            "user_id":     "voter_id",
            "user_gender": "voter_gender",
            "user_age":    "voter_age",
            "user_level":  "voter_level",
        })
        df = df.merge(df_voter, on="voter_id", how="left")

    # ---- JOIN 商品特征 ----
    df = df.merge(df_item, on="item_id", how="left")

    return df


# ═══════════════════════════════════════════════════════════════
# 错误记录 #1 — 2026-05-18
# ═══════════════════════════════════════════════════════════════
# 报错: KeyError: 'voter_id' (line 31: df.merge(df_voter, on="voter_id"))
# 原因: 不同分享表的列名不同：
#   - 训练集 (train / final_train): 有 voter_id 列
#   - 测试集 (prelim_test / final_test): 无 voter_id 列，只有 triple_id
#   测试集的 voter_id 是待预测目标，数据中不提供。
# 解决: 用 if "voter_id" in df.columns 判断，仅训练集做 voter 侧 JOIN。
#
# 错误记录 #2 — 2026-05-18
# 报错: NameError: name 'df_prelim_test' is not defined
# 原因: 直接在函数定义 cell 内跑验证代码，若 kernel 重置后只执行本 cell，
#   前面的数据加载 cell 未运行，df_prelim_test / df_user 等变量不存在。
# 解决: 验证代码加 try/except，NameError 时提示先跑前面的 cell。
# ═══════════════════════════════════════════════════════════════

# 尝试验证（依赖前面 cell 已执行；若未执行则提示）
try:
    print("🧪 对初赛测试集做合并验证...")
    df_test = merge_share_with_features(df_prelim_test, df_user, df_item)
    print(f"  合并后: {df_test.shape[0]:,} 行 × {df_test.shape[1]} 列")
    print(f"  列名: {list(df_test.columns)}")
    display(df_test.head(3))

    print("\n🧪 对训练集做合并验证...")
    df_test2 = merge_share_with_features(df_train.head(1000), df_user, df_item)
    print(f"  合并后: {df_test2.shape[0]:,} 行 × {df_test2.shape[1]} 列")
    print(f"  列名: {list(df_test2.columns)}")
    display(df_test2.head(3))

    del df_test, df_test2; gc.collect()
except NameError as e:
    print(f"⚠️ 变量未定义: {e}")
    print("请先执行本 notebook 前面的 cell（数据加载部分），再重新运行本 cell。")

🧪 对初赛测试集做合并验证...
  合并后: 118,424 行 × 11 列
  列名: ['triple_id', 'inviter_id', 'item_id', 'timestamp', 'inviter_gender', 'inviter_age', 'inviter_level', 'cate_id', 'cate_level1_id', 'brand_id', 'shop_id']


,triple_id,inviter_id,item_id,timestamp,inviter_gender,inviter_age,inviter_level,cate_id,cate_level1_id,brand_id,shop_id
0,0,e533ea22393b01170c9073ecd455aa1e,c825a3efd3b839d88f8bd9db9e4f7d75,2022-10-29 02:48:53,1,4,9,1623,16,1360044166,465965982
1,1,34d700d62e65c9c3bf10ad824751c035,e832e5e57045907c94c1b7c513eec868,2022-10-29 02:50:46,0,2,5,50010808,50010788,1616291913,33917004
2,2,278eddf6000cef49f5af15cc345258d1,cc3b15e06c0f51ff5dc1e0d4a5b64a45,2022-10-29 03:04:50,-1,-1,7,1623,16,4980787347,342462951



🧪 对训练集做合并验证...
  合并后: 1,000 行 × 14 列
  列名: ['inviter_id', 'item_id', 'voter_id', 'timestamp', 'inviter_gender', 'inviter_age', 'inviter_level', 'voter_gender', 'voter_age', 'voter_level', 'cate_id', 'cate_level1_id', 'brand_id', 'shop_id']


,inviter_id,item_id,voter_id,timestamp,inviter_gender,inviter_age,inviter_level,voter_gender,voter_age,voter_level,cate_id,cate_level1_id,brand_id,shop_id
0,5113da4ea98c5b38de164f4ee7a68d56,a3f617bc7082d6990ef58521e95a4cf5,6687b95557d3a031ec82a1dc35c9c8d7,2021-12-27 00:00:15,0,2,5,1,3,3,121400029,50011397,3650283,64996227
1,2fb4c533418044351f08b9a1e936ee0b,233cb3aa217fe3002238f31b1c71d5b8,680bec0f9f12393fb9b702cb8731fef1,2021-12-27 00:09:13,1,2,2,0,2,7,50010159,30,29534,232082092
2,5bbd4adf05e0607f6be4de1ec281fb4c,ba4f4367329239b424fcc6569274cb8e,d98cb64d067b471fe13affd32b2bb91c,2021-12-27 00:21:38,0,2,5,0,2,3,50011277,16,0,101581399


In [ ]:
# ============================================================
# 全量合并 4 个分享表 + 标记来源 + 落盘（实际上来说没啥用）
# ============================================================
#
# 整体逻辑:
#   1. 对 4 张原始分享表，逐张调用 merge_share_with_features()
#      → 每张表都做 3 次 LEFT JOIN (inviter画像 + voter画像 + 商品特征)
#   2. 给每张表打上 dataset 标签 (train / prelim_test / final_train / final_test)
#   3. 用 pd.concat() 纵向拼接为一张总表 (share_merged.pkl)
#   4. 同时保存各子集，方便后续按阶段独立取用
#
# ——— merge vs concat ———
#
# merge():  横向拼接 — 按 key 列做 JOIN，把右边表的新列"贴"到左边表旁边
#            类似 SQL: LEFT JOIN ... ON key
#            例: 分享表(4列) LEFT JOIN 用户表(3列) ON inviter_id → 7列
#            在这个项目里，封装在 merge_share_with_features() 函数中
#
# concat(): 纵向拼接 — 多张结构相同的表，行数叠加
#            类似 SQL: UNION ALL
#            例: 表A(60万行×16列) + 表B(12万行×16列) → 72万行×16列
#            参数 axis=0(默认) 纵向堆叠，axis=1 横向拼接
#
# ——— append() 已废弃，用 concat() 替代 ———
#
# 旧写法 (pandas < 2.0, 已废弃):
#   df_all = df_a.append(df_b).append(df_c)     # ← 链式调用，每次创建新对象
#
# 新写法:
#   df_all = pd.concat([df_a, df_b, df_c], ignore_index=True)  # ← 一次搞定
#
# concat 比逐次 append 好的原因:
#   - concat 一次性处理所有 DataFrame，只分配一次内存
#   - append 每次调用都创建新对象，链式调用产生多份中间拷贝
#   - 数据量大时，concat 明显更快
#
# ——— 列不一致时 concat 会怎么做? ———
#
# 训练集有 15 列 (含 voter_id + voter_gender + voter_age + voter_level)
# 测试集有 12 列 (无 voter 侧特征，只有 triple_id)
# concat 自动对齐列名 → 缺失的列填 NaN:
#
#            inviter_id  item_id  voter_id  ...  voter_gender  triple_id
#   train_0       A         X        B     ...       1           NaN
#   test_0        C         Y       NaN    ...      NaN          0
#                                ↑ 测试集没有 voter_id    ↑ 训练集没有 triple_id
#   → concat 后总表 16 列 (12+4 并集，双方独有的列都有)
#
# 为什么保存在 processed/ 而不放在 info/?
#   info/     — 原始竞赛数据 (只读，不改动)
#   processed/ — 加工后的中间产物 (本项目内部使用，随时可重建)
# ============================================================

dfs_to_merge = [
    (df_train,        "train"),
    (df_prelim_test,  "prelim_test"),
    (df_final_train,  "final_train"),
    (df_final_test,   "final_test"),
]

merged_dfs = []

for df_src, label in dfs_to_merge:
    print(f"合并 {label} ({df_src.shape[0]:,} 行)...", end=" ", flush=True)
    # ——— merge: 横向拼接，LEFT JOIN 用户画像 + 商品特征 ———
    #     训练集有 voter_id → 拼 inviter+voter+item 三侧特征 (14列)
    #     测试集无 voter_id → 只拼 inviter+item 两侧特征 (11列)
    df_merged = merge_share_with_features(df_src, df_user, df_item)
    df_merged["dataset"] = label           # 标记来源，便于后续按阶段筛选，同时提供数据溯源
    merged_dfs.append(df_merged)
    print(f"→ {df_merged.shape[0]:,} 行 × {df_merged.shape[1]} 列")

# ——— concat: 纵向拼接，4 张结构相似的表堆叠为一张总表 ———
#     ignore_index=True: 重新编号 0 ~ N-1
#     如果不设: 每个子集的原始 index 被保留, 会出现重复索引
#     例: df_train index 0~602678, df_final_test 也有 index 0~115413
#     → df_all.loc[0] 会返回多行 (pandas 不报错但结果不符合预期)
print("\n拼接全部数据集...")
df_all = pd.concat(merged_dfs, ignore_index=True)
print(f"总表: {df_all.shape[0]:,} 行 × {df_all.shape[1]} 列")

# ============================================================
# 保存为 pickle — pandas 原生二进制格式
# ============================================================
# 注释写"Parquet"但实际用 pickle 的原因:
#   pickle:  pandas 原生，读写最快，列类型完整保留 (datetime、category 等)
#   parquet: 跨语言通用，压缩率更高，但某些 pandas 特殊类型会丢失
#   这个项目只在 Python 内部使用 → pickle 就够了
OUTPUT_DIR = BASE_DIR / "processed"
OUTPUT_DIR.mkdir(exist_ok=True)

PARQUET_PATH = OUTPUT_DIR / "share_merged.pkl"
df_all.to_pickle(PARQUET_PATH)
print(f"\n✅ 已保存至: {PARQUET_PATH}")

# 同时保存各子集，方便后续按阶段独立使用
# 比如: build_train_matrix.py 只需要 share_train + share_final_train
#       不需要加载整张 95 万行的总表，节省内存
for df_merged in merged_dfs:
    label = df_merged["dataset"].iloc[0]
    path = OUTPUT_DIR / f"share_{label}.pkl"
    df_merged.to_pickle(path)
    print(f"  子集保存: {path}")

# 查看 dataset 分布 — 确认 4 个子集数量正常
print("\n📊 各数据集记录数:")
print(df_all["dataset"].value_counts().to_string())
# 期望输出:
#   train          602679  (初赛训练, 2021-12 ~ 2022-10)
#   prelim_test    118424  (初赛测试, 2022-10 ~ 2022-12)
#   final_train    118424  (决赛训练, 2022-10 ~ 2022-12)
#   final_test     115414  (决赛测试, 2022-12 ~ 2023-02)

# 释放原始 share 表的中间变量（合并后不再需要）
# df_all 已经保存为 pkl，后续按需加载即可，内存里的四张原表可以扔掉
del df_train, df_prelim_test, df_final_train, df_final_test
del merged_dfs
gc.collect()
print("\n✅ 1.1 数据加载与合并 — 完成")